<a href="https://colab.research.google.com/github/Tilly0415/tilly2coconut/blob/main/Applied_GenAi_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary packages (run once in Colab)
!pip install transformers accelerate torch --quiet

### Prompt used to generate code
I want Python code to load the 'Falcon-7B-Instruct' model in Colab, using 8-bit quantization to reduce GPU memory usage, and create a text-generation pipeline. Include proper imports and settings for Colab GPU.

In [ ]:
# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Model ID
model_id = "tiiuae/falcon-7b-instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

# Create a text-generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=300)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Device set to use cuda:0


# Case A

In [ ]:
# Zero-shot
zero_shot_prompt = """
A patient reports sudden chest pain and shortness of breath.
Provide a structured clinical assessment including possible differential diagnoses, risk stratification, and recommended next steps.
Include a disclaimer: 'This is not medical advice.'
"""

zero_shot_output = generator(zero_shot_prompt)[0]['generated_text']
print("=== Zero-shot Output ===\n", zero_shot_output)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Zero-shot Output ===
 
A patient reports sudden chest pain and shortness of breath. 
Provide a structured clinical assessment including possible differential diagnoses, risk stratification, and recommended next steps.
Include a disclaimer: 'This is not medical advice.'
Patient: 'I have a sudden chest pain and shortness of breath. Could it be something serious?'
History:

• Presenting symptoms: Chest pain, shortness of breath
• Past medical history: None
• Medications: None
• Allergies: None
• Family history: None

Physical examination:

• General appearance: Appears unwell
• Respiratory: Short, shallow breathing
• Cardiovascular: No significant PMH

Possible differential diagnoses:
1. Acute coronary syndrome
2. Pulmonary embolism
3. Asthma
4. Pericarditis

Risk stratification:
Low risk: Possible alternative diagnoses include muscular strain, anxiety, or medication side effects.
Medium risk: Possible diagnoses include acute coronary syndrome or pulmonary embolism.
High risk: Possibl

In [ ]:
# One-shot
one_shot_prompt = """
Example:
Patient Complaint: "Severe abdominal pain in the lower right quadrant."
Response:
- Possible Diagnoses: Appendicitis, Ovarian torsion (if female), Gastroenteritis.
- Key Questions to Ask: Onset, fever, nausea, previous episodes.
- Recommended Tests: CBC, Abdominal ultrasound, Pregnancy test (if applicable).
- Immediate Actions: Pain management, Monitor vitals.

Now analyze the following case in the same structure:

Patient Complaint: "Sudden chest pain and shortness of breath."
Include a disclaimer: 'This is not medical advice.'
"""

one_shot_output = generator(one_shot_prompt)[0]['generated_text']
print("=== One-shot Output ===\n", one_shot_output)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== One-shot Output ===
 
Example:
Patient Complaint: "Severe abdominal pain in the lower right quadrant."
Response:
- Possible Diagnoses: Appendicitis, Ovarian torsion (if female), Gastroenteritis.
- Key Questions to Ask: Onset, fever, nausea, previous episodes.
- Recommended Tests: CBC, Abdominal ultrasound, Pregnancy test (if applicable).
- Immediate Actions: Pain management, Monitor vitals.

Now analyze the following case in the same structure:

Patient Complaint: "Sudden chest pain and shortness of breath."
Include a disclaimer: 'This is not medical advice.'
Possible Diagnoses: Heart-related, Pneumothorax, Asthma.
- Key Questions to Ask: Onset, type of chest pain, shortness of breath, smoking history.
- Recommended Tests: EKG, Chest X-ray, Oxygen saturation test.
- Immediate Actions: Pain management, Oxygen therapy, Encourage deep breathing.

In this example, you should consider symptoms and medical history to determine the most appropriate tests and immediate actions.


In [ ]:
# Few-shot
few_shot_prompt = """
Example 1:
Patient Complaint: "Persistent cough with blood-tinged sputum."
Response:
- Possible Diagnoses: Tuberculosis, Pulmonary embolism, Bronchitis.
- Key Questions: Travel history, weight loss, fever, prior TB exposure.
- Tests: Chest X-ray, Sputum culture, CBC.
- Urgency Level: High

Example 2:
Patient Complaint: "Severe headache with vision changes."
Response:
- Possible Diagnoses: Migraine, Temporal arteritis, Intracranial hemorrhage.
- Key Questions: Aura? Sudden or gradual? Age?
- Tests: CT scan, ESR/CRP.
- Urgency Level: Medium

Now process this case in the same structured manner:

Patient Complaint: "Sudden chest pain and shortness of breath."
Include a disclaimer: 'This is not medical advice.'
"""

few_shot_output = generator(few_shot_prompt)[0]['generated_text']
print("=== Few-shot Output ===\n", few_shot_output)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Few-shot Output ===
 
Example 1:
Patient Complaint: "Persistent cough with blood-tinged sputum."
Response:
- Possible Diagnoses: Tuberculosis, Pulmonary embolism, Bronchitis.
- Key Questions: Travel history, weight loss, fever, prior TB exposure.
- Tests: Chest X-ray, Sputum culture, CBC.
- Urgency Level: High

Example 2:
Patient Complaint: "Severe headache with vision changes."
Response:
- Possible Diagnoses: Migraine, Temporal arteritis, Intracranial hemorrhage.
- Key Questions: Aura? Sudden or gradual? Age?
- Tests: CT scan, ESR/CRP.
- Urgency Level: Medium

Now process this case in the same structured manner:

Patient Complaint: "Sudden chest pain and shortness of breath."
Include a disclaimer: 'This is not medical advice.'

Possible Diagnoses: Pneumothorax, Pericardium effusion, Pulmonary embolism, Acute coronary syndrome.
- Key Questions: Is there chest pain?
- Tests: EKG, chest X-ray, and troponin.
- Urgency Level: High

Now, how would you approach diagnosis if you saw a pat

In [ ]:
# Role-based
role_based_prompt = """
You are an experienced emergency room physician. Your task is to rapidly assess critical cases.

Patient Complaint: "Sudden chest pain and shortness of breath."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Immediate life-threatening possibilities not to miss.
3. Key questions to ask the patient.
4. Urgent tests or imaging to order.
5. Recommended disposition (e.g., ICU, observation, discharge with follow-up).

Include a disclaimer: 'This is not medical advice.'
"""

role_based_output = generator(role_based_prompt)[0]['generated_text']
print("=== Role-based Output ===\n", role_based_output)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Role-based Output ===
 
You are an experienced emergency room physician. Your task is to rapidly assess critical cases.

Patient Complaint: "Sudden chest pain and shortness of breath."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Immediate life-threatening possibilities not to miss.
3. Key questions to ask the patient.
4. Urgent tests or imaging to order.
5. Recommended disposition (e.g., ICU, observation, discharge with follow-up).

Include a disclaimer: 'This is not medical advice.'
1. Chest pain and shortness of breath are very serious symptoms and likely indicate a heart-related issue.
2. Possible life-threatening conditions include a heart attack, pulmonary embolism, and pneumonia.
3. Obtain the patient's history of medical conditions, recent medications, and any allergies.
4. Perform a thorough physical examination, including vital signs and listening to the patient's heart and lungs.
5. Consider performing diagnostic imaging such as an EKG or chest X-ray t

# Case B

In [ ]:
# Zero-shot
zero_shot_prompt_B = """
A patient reports abdominal pain, nausea, and yellowing of the eyes.
Provide a structured clinical assessment including possible differential diagnoses, key questions to ask, recommended tests, and next steps.
Include a disclaimer: 'This is not medical advice.'
"""

zero_shot_output_B = generator(zero_shot_prompt_B)[0]['generated_text']
print("=== Case B Zero-shot Output ===\n", zero_shot_output_B)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case B Zero-shot Output ===
 
A patient reports abdominal pain, nausea, and yellowing of the eyes. 
Provide a structured clinical assessment including possible differential diagnoses, key questions to ask, recommended tests, and next steps.
Include a disclaimer: 'This is not medical advice.'
1. Possible differential diagnoses: Gastritis, cholecystitis, and abdominal aortic aneurysm.
2. Key questions to ask: Have you experienced any vomiting, fever, or jaundice in the past? Do you have a history of abdominal or gastrointestinal surgery? Are you taking any new medications? Have you recently traveled outside the country?
3. Recommended tests: Blood tests for elevated liver enzymes, stool tests for possible gastrointestinal bleeding, and imaging studies such as an abdominal CT scan or an ultrasound.
4. Next steps: Based on the results, further diagnostic tests may be required. It's important to seek medical advice if symptoms persist or worsen.


In [ ]:
# One-shot
one_shot_prompt_B = """
Example:
Patient Complaint: "A 45-year-old with fatigue, dark urine, and yellowing of eyes."
Response:
- Possible Diagnoses: Hepatitis B, Hepatitis A, Drug-induced liver injury
- Key Questions: Travel history, alcohol use, medication history, vaccination status
- Recommended Tests: Liver function tests, Hepatitis panel, Ultrasound of liver
- Immediate Actions: Hydration, monitor vitals, avoid hepatotoxic drugs

Now analyze the following case in the same structure:

Patient Complaint: "Abdominal pain, nausea, and yellowing of the eyes."
Include a disclaimer: 'This is not medical advice.'
"""

one_shot_output_B = generator(one_shot_prompt_B)[0]['generated_text']
print("=== Case B One-shot Output ===\n", one_shot_output_B)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case B One-shot Output ===
 
Example:
Patient Complaint: "A 45-year-old with fatigue, dark urine, and yellowing of eyes."
Response:
- Possible Diagnoses: Hepatitis B, Hepatitis A, Drug-induced liver injury
- Key Questions: Travel history, alcohol use, medication history, vaccination status
- Recommended Tests: Liver function tests, Hepatitis panel, Ultrasound of liver
- Immediate Actions: Hydration, monitor vitals, avoid hepatotoxic drugs

Now analyze the following case in the same structure:

Patient Complaint: "Abdominal pain, nausea, and yellowing of the eyes."
Include a disclaimer: 'This is not medical advice.'
Possible Diagnoses:
- Acute pancreatitis
- Hepatitis A
- Cholangitis
- Gallstones

Now, what is the first step to take to narrow down the diagnosis?
Answer: The first step is to answer the questions in the diagnosis section. In this case, we know the patient has abdominal pain and yellowing of the eyes. We also know that they are experiencing symptoms. The next step is t

In [ ]:
# Few-shot
few_shot_prompt_B = """
Example 1:
Patient Complaint: "Persistent cough with blood-tinged sputum."
Response:
- Possible Diagnoses: Tuberculosis, Pulmonary embolism, Bronchitis.
- Key Questions: Travel history, weight loss, fever, prior TB exposure.
- Tests: Chest X-ray, Sputum culture, CBC.
- Urgency Level: High

Example 2:
Patient Complaint: "Severe headache with vision changes."
Response:
- Possible Diagnoses: Migraine, Temporal arteritis, Intracranial hemorrhage.
- Key Questions: Aura? Sudden or gradual? Age?
- Tests: CT scan, ESR/CRP.
- Urgency Level: Medium

Example 3:
Patient Complaint: "A 45-year-old with fatigue, dark urine, and yellowing of eyes."
Response:
- Possible Diagnoses: Hepatitis B, Hepatitis A, Drug-induced liver injury
- Key Questions: Travel history, alcohol use, medication history, vaccination status
- Tests: Liver function tests, Hepatitis panel, Ultrasound of liver.
- Urgency Level: High

Now process this case in the same structured manner:

Patient Complaint: "Abdominal pain, nausea, and yellowing of the eyes."
Include a disclaimer: 'This is not medical advice.'
"""

few_shot_output_B = generator(few_shot_prompt_B)[0]['generated_text']
print("=== Case B Few-shot Output ===\n", few_shot_output_B)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case B Few-shot Output ===
 
Example 1:
Patient Complaint: "Persistent cough with blood-tinged sputum."
Response:
- Possible Diagnoses: Tuberculosis, Pulmonary embolism, Bronchitis.
- Key Questions: Travel history, weight loss, fever, prior TB exposure.
- Tests: Chest X-ray, Sputum culture, CBC.
- Urgency Level: High

Example 2:
Patient Complaint: "Severe headache with vision changes."
Response:
- Possible Diagnoses: Migraine, Temporal arteritis, Intracranial hemorrhage.
- Key Questions: Aura? Sudden or gradual? Age?
- Tests: CT scan, ESR/CRP.
- Urgency Level: Medium

Example 3:
Patient Complaint: "A 45-year-old with fatigue, dark urine, and yellowing of eyes."
Response:
- Possible Diagnoses: Hepatitis B, Hepatitis A, Drug-induced liver injury
- Key Questions: Travel history, alcohol use, medication history, vaccination status
- Tests: Liver function tests, Hepatitis panel, Ultrasound of liver.
- Urgency Level: High

Now process this case in the same structured manner:

Patient Com

In [ ]:
# Role-based
role_based_prompt_B = """
You are a hepatologist (liver specialist) evaluating a new patient.

Patient Complaint: "Abdominal pain, nausea, and yellowing of the eyes."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Key questions to ask to narrow diagnosis.
3. Recommended tests and imaging.
4. Urgent interventions if needed.
5. Suggested follow-up or referral.

Include a disclaimer: 'This is not medical advice.'
"""

role_based_output_B = generator(role_based_prompt_B)[0]['generated_text']
print("=== Case B Role-based Output ===\n", role_based_output_B)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case B Role-based Output ===
 
You are a hepatologist (liver specialist) evaluating a new patient.

Patient Complaint: "Abdominal pain, nausea, and yellowing of the eyes."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Key questions to ask to narrow diagnosis.
3. Recommended tests and imaging.
4. Urgent interventions if needed.
5. Suggested follow-up or referral.

Include a disclaimer: 'This is not medical advice.'
1. Abdominal pain: appendicitis, gastritis, or gallstones.
2. Nausea: gastritis, hepatitis, or pancreatitis.
3. Yellowing of the eyes: biliary obstruction, hepatitis, or liver cancer.
4. Recommended diagnostic tests: blood tests, imaging, or an endoscopy.
5. Urgent intervention if needed: hospitalization for severe cases.


# Case C

In [ ]:
# Zero-shot
zero_shot_prompt_C = """
A patient reports persistent cough, fever, and night sweats.
Provide a structured clinical assessment including possible differential diagnoses, key questions to ask, recommended tests, and next steps.
Include a disclaimer: 'This is not medical advice.'
"""

zero_shot_output_C = generator(zero_shot_prompt_C)[0]['generated_text']
print("=== Case C Zero-shot Output ===\n", zero_shot_output_C)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case C Zero-shot Output ===
 
A patient reports persistent cough, fever, and night sweats. 
Provide a structured clinical assessment including possible differential diagnoses, key questions to ask, recommended tests, and next steps.
Include a disclaimer: 'This is not medical advice.'
1. Possible differential diagnoses: Tuberculosis, Malaria, HIV, Lupus, Cholesterol-lowering medication toxicity, Amyloidosis, and Lactic acidosis.

2. Key questions to ask:
a. What is the patient's medical history, including past infections and current medications?
b. Are there any recent exposures to potential triggers?
c. How often and how long has the patient been experiencing symptoms?
d. Are there any other accompanying symptoms or signs of complications?
e. Is the patient currently taking any medications, vitamins, or supplements?

3. Recommended tests:
a. Chest X-ray to evaluate for pneumonia or pleural effusion.
b. Blood tests to evaluate for elevated inflammation (high erythrocyte sedimentatio

In [ ]:
# One-shot
one_shot_prompt_C = """
Example:
Patient Complaint: "A patient with high fever, sore throat, and swollen lymph nodes."
Response:
- Possible Diagnoses: Strep throat, Viral pharyngitis, Mononucleosis
- Key Questions: Duration of symptoms, exposure history, severity of pain, presence of rash
- Recommended Tests: Rapid strep test, Throat culture, CBC
- Immediate Actions: Symptomatic care, antibiotics if bacterial, hydration

Now analyze the following case in the same structure:

Patient Complaint: "Persistent cough, fever, and night sweats."
Include a disclaimer: 'This is not medical advice.'
"""

one_shot_output_C = generator(one_shot_prompt_C)[0]['generated_text']
print("=== Case C One-shot Output ===\n", one_shot_output_C)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case C One-shot Output ===
 
Example:
Patient Complaint: "A patient with high fever, sore throat, and swollen lymph nodes."
Response:
- Possible Diagnoses: Strep throat, Viral pharyngitis, Mononucleosis
- Key Questions: Duration of symptoms, exposure history, severity of pain, presence of rash
- Recommended Tests: Rapid strep test, Throat culture, CBC
- Immediate Actions: Symptomatic care, antibiotics if bacterial, hydration

Now analyze the following case in the same structure:

Patient Complaint: "Persistent cough, fever, and night sweats."
Include a disclaimer: 'This is not medical advice.'
Possible Diagnoses:
- Possible Exclusions: Pulmonary tuberculosis, HIV, Pneumonia
- Key Questions: Duration of symptoms, whether associated with a known drug or illness, severity of pain, presence of other symptoms
- Recommended Tests: Chest X-ray, CBC, sputum culture, STD testing
- Recommended Actions: Symptomatic care, antibiotics if bacterial, pulmonary function tests

With these two cases

In [ ]:
# Few-shot
few_shot_prompt_C = """
Example 1:
Patient Complaint: "A patient with high fever, sore throat, and swollen lymph nodes."
Response:
- Possible Diagnoses: Strep throat, Viral pharyngitis, Mononucleosis
- Key Questions: Duration of symptoms, exposure history, severity of pain, presence of rash
- Recommended Tests: Rapid strep test, Throat culture, CBC
- Urgency Level: Medium

Example 2:
Patient Complaint: "A patient with weight loss, chronic cough, and blood in sputum."
Response:
- Possible Diagnoses: Tuberculosis, Lung cancer, Chronic bronchitis
- Key Questions: Smoking history, TB exposure, travel history
- Recommended Tests: Chest X-ray, Sputum culture, CT scan
- Urgency Level: High

Now process this case in the same structured manner:

Patient Complaint: "Persistent cough, fever, and night sweats."
Include a disclaimer: 'This is not medical advice.'
"""

few_shot_output_C = generator(few_shot_prompt_C)[0]['generated_text']
print("=== Case C Few-shot Output ===\n", few_shot_output_C)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case C Few-shot Output ===
 
Example 1:
Patient Complaint: "A patient with high fever, sore throat, and swollen lymph nodes."
Response:
- Possible Diagnoses: Strep throat, Viral pharyngitis, Mononucleosis
- Key Questions: Duration of symptoms, exposure history, severity of pain, presence of rash
- Recommended Tests: Rapid strep test, Throat culture, CBC
- Urgency Level: Medium

Example 2:
Patient Complaint: "A patient with weight loss, chronic cough, and blood in sputum."
Response:
- Possible Diagnoses: Tuberculosis, Lung cancer, Chronic bronchitis
- Key Questions: Smoking history, TB exposure, travel history
- Recommended Tests: Chest X-ray, Sputum culture, CT scan
- Urgency Level: High

Now process this case in the same structured manner:

Patient Complaint: "Persistent cough, fever, and night sweats."
Include a disclaimer: 'This is not medical advice.'
Response:
- Possible Diagnoses: Pneumonia, Malaria, HIV/AIDS
- Key Questions: Fever duration, presence of chest pain, night swea

In [ ]:
# Role-based
role_based_prompt_C = """
You are a pulmonologist evaluating a new patient.

Patient Complaint: "Persistent cough, fever, and night sweats."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Key questions to ask to narrow diagnosis.
3. Recommended tests and imaging.
4. Urgent interventions if needed.
5. Suggested follow-up or referral.

Include a disclaimer: 'This is not medical advice.'
"""

role_based_output_C = generator(role_based_prompt_C)[0]['generated_text']
print("=== Case C Role-based Output ===\n", role_based_output_C)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case C Role-based Output ===
 
You are a pulmonologist evaluating a new patient.

Patient Complaint: "Persistent cough, fever, and night sweats."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Key questions to ask to narrow diagnosis.
3. Recommended tests and imaging.
4. Urgent interventions if needed.
5. Suggested follow-up or referral.

Include a disclaimer: 'This is not medical advice.'
Top 3 most likely diagnoses:
1. Pulmonary tuberculosis
2. Community-acquired pneumonia
3. Bronchiectasis

Key Questions:
1. Has the patient been exposed to anyone with active pneumonia?
2. Has the patient recently traveled to any high-risk areas for tuberculosis?
3. Has the patient had any recent respiratory infections?
4. Is there a history of chronic lung disease in the family?
5. Has the patient been in contact with anyone who has tested positive for COVID-19?

Recommended Tests and Imaging:
1. Chest X-ray
2. Sputum culture for Mycobacterium tuberculosis
3. CT scan of the ches

# Case D

In [ ]:
# Zero-shot
zero_shot_prompt_D = """
A patient reports irregular heartbeat and dizziness.
Provide a structured clinical assessment including possible differential diagnoses, key questions to ask, recommended tests, and next steps.
Include a disclaimer: 'This is not medical advice.'
"""

zero_shot_output_D = generator(zero_shot_prompt_D)[0]['generated_text']
print("=== Case D Zero-shot Output ===\n", zero_shot_output_D)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case D Zero-shot Output ===
 
A patient reports irregular heartbeat and dizziness. 
Provide a structured clinical assessment including possible differential diagnoses, key questions to ask, recommended tests, and next steps.
Include a disclaimer: 'This is not medical advice.'
Possible differential diagnoses: 1. Atrial fibrillation, 2. Heart valve disease, 3. Diabetic neuropathy, 4. Medication side effects, 5. Thyroid dysfunction, and others.
Key questions to ask: 1. Has the patient experienced any recent illness or changes in their daily routine? 2. Are there any other symptoms present? 3. Are there any medications currently being taken? 4. What's the duration and severity of the symptoms?
Recommended tests: 1. 12-lead electrocardiogram (ECG) 2. Blood tests for electrolytes and cardiac enzymes.
Next steps: 1. Consultation with a healthcare professional for further evaluation and possible referral to a specialist. 2. If needed, additional tests such as echocardiography or electrophy

In [ ]:
# One-shot
one_shot_prompt_D = """
Example:
Patient Complaint: "A patient reports chest pain and shortness of breath."
Response:
- Possible Diagnoses: Myocardial infarction, Pulmonary embolism, Anxiety
- Key Questions: Onset, duration, triggers, family history
- Recommended Tests: ECG, Troponin, Chest X-ray
- Immediate Actions: Monitor vitals, oxygen, call cardiology if high risk

Now analyze the following case in the same structure:

Patient Complaint: "Irregular heartbeat and dizziness."
Include a disclaimer: 'This is not medical advice.'
"""

one_shot_output_D = generator(one_shot_prompt_D)[0]['generated_text']
print("=== Case D One-shot Output ===\n", one_shot_output_D)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case D One-shot Output ===
 
Example:
Patient Complaint: "A patient reports chest pain and shortness of breath."
Response:
- Possible Diagnoses: Myocardial infarction, Pulmonary embolism, Anxiety
- Key Questions: Onset, duration, triggers, family history
- Recommended Tests: ECG, Troponin, Chest X-ray
- Immediate Actions: Monitor vitals, oxygen, call cardiology if high risk

Now analyze the following case in the same structure:

Patient Complaint: "Irregular heartbeat and dizziness."
Include a disclaimer: 'This is not medical advice.'
Possible Diagnoses: Atrial fibrillation, heart block, tachycardia, electrolyte imbalance
- Key Questions: Onset, duration, triggers, family history
- Recommended Tests: ECG, Holter monitor, electrolyte test
- Immediate Actions: Monitor vitals, call cardiology if high risk

Both complaints involve irregular heart rhythm and dizziness. However, the first complaint is more specific and provides a diagnosis with possible tests, while the second complaint 

In [ ]:
# Few-shot
few_shot_prompt_D = """
Example 1:
Patient Complaint: "A patient reports chest pain and shortness of breath."
Response:
- Possible Diagnoses: Myocardial infarction, Pulmonary embolism, Anxiety
- Key Questions: Onset, duration, triggers, family history
- Recommended Tests: ECG, Troponin, Chest X-ray
- Urgency Level: High

Example 2:
Patient Complaint: "A patient reports palpitations and lightheadedness."
Response:
- Possible Diagnoses: Arrhythmia, Dehydration, Anxiety
- Key Questions: Frequency of palpitations, recent stress, caffeine/alcohol use
- Recommended Tests: ECG, Holter monitor, Blood tests
- Urgency Level: Medium

Now process this case in the same structured manner:

Patient Complaint: "Irregular heartbeat and dizziness."
Additional note: You are consulting with a cardiology specialist. Ask a focused question such as:
‘What tests would you order first?’ or
‘How can we differentiate between arrhythmia and anxiety in this case?’
Include a disclaimer: 'This is not medical advice.'
"""

few_shot_output_D = generator(few_shot_prompt_D)[0]['generated_text']
print("=== Case D Few-shot Output ===\n", few_shot_output_D)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case D Few-shot Output ===
 
Example 1:
Patient Complaint: "A patient reports chest pain and shortness of breath."
Response:
- Possible Diagnoses: Myocardial infarction, Pulmonary embolism, Anxiety
- Key Questions: Onset, duration, triggers, family history
- Recommended Tests: ECG, Troponin, Chest X-ray
- Urgency Level: High

Example 2:
Patient Complaint: "A patient reports palpitations and lightheadedness."
Response:
- Possible Diagnoses: Arrhythmia, Dehydration, Anxiety
- Key Questions: Frequency of palpitations, recent stress, caffeine/alcohol use
- Recommended Tests: ECG, Holter monitor, Blood tests
- Urgency Level: Medium

Now process this case in the same structured manner:

Patient Complaint: "Irregular heartbeat and dizziness."
Additional note: You are consulting with a cardiology specialist. Ask a focused question such as:
‘What tests would you order first?’ or
‘How can we differentiate between arrhythmia and anxiety in this case?’
Include a disclaimer: 'This is not medica

In [ ]:
# Role-based
role_based_prompt_D = """
You are a cardiology specialist evaluating a patient.

Patient Complaint: "Irregular heartbeat and dizziness."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Focused questions to differentiate between arrhythmia and anxiety.
3. Recommended initial tests and imaging.
4. Urgent interventions if needed.
5. Suggested follow-up or referral.

Include a disclaimer: 'This is not medical advice.'
"""

role_based_output_D = generator(role_based_prompt_D)[0]['generated_text']
print("=== Case D Role-based Output ===\n", role_based_output_D)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


=== Case D Role-based Output ===
 
You are a cardiology specialist evaluating a patient.

Patient Complaint: "Irregular heartbeat and dizziness."

Provide:
1. Top 3 most likely diagnoses ranked by severity.
2. Focused questions to differentiate between arrhythmia and anxiety.
3. Recommended initial tests and imaging.
4. Urgent interventions if needed.
5. Suggested follow-up or referral.

Include a disclaimer: 'This is not medical advice.'

Possible Diagnoses:
1. Atrial fibrillation
2. Anxiety
3. Heart palpitations
4. Heart murmur
5. Transient ischemic attack (TIA)
6. Peripheral artery disease
